## PREPARATIONS

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import psycopg2
import psycopg2.extras as extras
import logging
import pandas as pd
import numpy as np
import ast

from config import DB_CONFIG
from db_tools import execute_query

## READ CSV + PARSING (SKIP)

In [3]:
df = pd.read_csv("tmdb_5000_movies.csv")
cols_to_clean = ['genres', 'keywords', 'production_companies', 'production_countries', 'spoken_languages']
def parse_json_column(column_value):
    try:
        parsed_list = ast.literal_eval(column_value)
        return [item['name'] for item in parsed_list]
    except (ValueError, SyntaxError):
        return []
for col in cols_to_clean:
    df[col] = df[col].apply(parse_json_column)
    df[col] = df[col].apply(lambda x: ', '.join(x))

## CONNECTING TO DB

In [4]:
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

def test_new_connection():
    try:
        conn = psycopg2.connect(**DB_CONFIG)
        logging.info("Successfully connected to movies_db")
        conn.close()
    except Exception as e:

        logging.error(f"Error: {e}")

test_new_connection()

INFO: Successfully connected to movies_db


## TABLE CREATION

In [5]:
from queries import CREATE_MOVIES_TABLE
execute_query(CREATE_MOVIES_TABLE)

## HEAD HTML

In [6]:
df.head().to_html('table.html')

## SAVING TABLE FROM DATAFRAME 

In [7]:
df.to_csv("tmdb_5000_movies_clean.csv", index=False)

## CONVERTING TO BIGINT

In [8]:
from queries import TO_BIGINT
#execute_query(TO_BIGINT)

## DATA INSERTION

In [9]:
from db_tools import insert_dataframe
table_name = 'movies'
columns_for_db = ['title','release_date','genres','vote_average','vote_count','runtime','budget','revenue']
df_prepared = df[columns_for_db]
df_prepared.head()
insert_dataframe(df_prepared, 'movies')

,title,release_date,genres,vote_average,vote_count,runtime,budget,revenue
0,Avatar,2009-12-10,"Action, Adventure, Fantasy, Science Fiction",7.2,11800,162.0,237000000,2787965087
1,Pirates of the Caribbean: At World's End,2007-05-19,"Adventure, Fantasy, Action",6.9,4500,169.0,300000000,961000000
2,Spectre,2015-10-26,"Action, Adventure, Crime",6.3,4466,148.0,245000000,880674609
3,The Dark Knight Rises,2012-07-16,"Action, Crime, Drama, Thriller",7.6,9106,165.0,250000000,1084939099
4,John Carter,2012-03-07,"Action, Adventure, Science Fiction",6.1,2124,132.0,260000000,284139100


## SAVING CSV FROM SQL

In [10]:
from queries import TO_CSV
execute_query(TO_CSV)

## CREATING RELATIONS (3NF)

In [11]:
from queries import CREATE_GENRES_TABLE
execute_query(CREATE_GENRES_TABLE)

In [12]:
from queries import INSERT_INTO_GENRES
execute_query(INSERT_INTO_GENRES)

In [13]:
from queries import CREATE_GENRE_MOVIE_TABLE
execute_query(CREATE_GENRE_MOVIE_TABLE)

In [14]:
from queries import CROSS_TABLE
execute_query(CROSS_TABLE)

In [15]:
from queries import DELETE_COLUMN
execute_query(DELETE_COLUMN)